Write a Python program that takes as input a file containing DNA sequences in multi-FASTA format, and computes the answers to the following questions. You can choose to write one program with multiple functions to answer these questions, or you can write several programs to address them. We will provide a multi-FASTA file for you, and you will run your program to answer the exam questions. 

While developing your program(s), please use the following example file to test your work: 
dna.example.fasta

You'll be given a different input file to launch the exam itself.

Here are the questions your program needs to answer. The quiz itself contains the specific multiple-choice questions you need to answer for the file you will be provided.

(1) How many records are in the file? A record in a FASTA file is defined as a single-line header, followed by lines of sequence data. The header line is distinguished from the sequence data by a greater-than (">") symbol in the first column. The word following the ">" symbol is the identifier of the sequence, and the rest of the line is an optional description of the entry. There should be no space between the ">" and the first letter of the identifier. 

(2) What are the lengths of the sequences in the file? What is the longest sequence and what is the shortest sequence? Is there more than one longest or shortest sequence? What are their identifiers? 

(3) In molecular biology, a reading frame is a way of dividing the DNA sequence of nucleotides into a set of consecutive, non-overlapping triplets (or codons). Depending on where we start, there are six possible reading frames: three in the forward (5' to 3') direction and three in the reverse (3' to 5'). For instance, the three possible forward reading frames for the sequence AGGTGACACCGCAAGCCTTATATTAGC are: 

AGG TGA CAC CGC AAG CCT TAT ATT AGC

A GGT GAC ACC GCA AGC CTT ATA TTA GC

AG GTG ACA CCG CAA GCC TTA TAT TAG C 

These are called reading frames 1, 2, and 3 respectively. An open reading frame (ORF) is the part of a reading frame that has the potential to encode a protein. It starts with a start codon (ATG), and ends with a stop codon (TAA, TAG or TGA). For instance, ATGAAATAG is an ORF of length 9.

Given an input reading frame on the forward strand (1, 2, or 3) your program should be able to identify all ORFs present in each sequence of the FASTA file, and answer the following questions: what is the length of the longest ORF in the file? What is the identifier of the sequence containing the longest ORF? For a given sequence identifier, what is the longest ORF contained in the sequence represented by that identifier? What is the starting position of the longest ORF in the sequence that contains it? The position should indicate the character number in the sequence. For instance, the following ORF in reading frame 1:

\>sequence1

ATGCCCTAG

starts at position 1.

Note that because the following sequence:

\>sequence2

ATGAAAAAA

does not have any stop codon in reading frame 1, we do not consider it to be an ORF in reading frame 1. 

(4) A repeat is a substring of a DNA sequence that occurs in multiple copies (more than one) somewhere in the sequence. Although repeats can occur on both the forward and reverse strands of the DNA sequence, we will only consider repeats on the forward strand here. Also we will allow repeats to overlap themselves. For example, the sequence ACACA contains two copies of the sequence ACA - once at position 1 (index 0 in Python), and once at position 3. Given a length n, your program should be able to identify all repeats of length n in all sequences in the FASTA file. Your program should also determine how many times each repeat occurs in the file, and which is the most frequent repeat of a given length.

In [113]:
from pprint import pprint

In [114]:
def extract_dna_seqs(file_name):
    '''The function saves the text in a FASTA file into a string.'''
    try:
        with open(file_name, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("File was not found or the file name is misspelled.")
        return None
    

def create_dna_dict(file_name):
    dna_seqs = extract_dna_seqs(file_name)
    dna_seq_dict = {}
    for line in dna_seqs.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            identifier = line[1:]
            dna_seq_dict[identifier] = ''
            continue
    
        dna_seq_dict[identifier] += line

    return dna_seq_dict

dna_seqs = create_dna_dict("dna.example.fasta")
print(dna_seqs)
len(dna_seqs)


{'gi|142022655|gb|EQ086233.1|43 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': 'TCGGGCGAAGGCGGCAGCAAGTCGTCCACGCGCAGCGCGGCACCGCGGGCCTCTGCCGTGCGCTGCTTGGCCATGGCCTCCAGCGCACCGATCGGATCAAAGCCGCTGAAGCCTTCGCGCATCAGGCGGCCATAGTTGGCGCCAGTGACCGTACCAACCGCCTTGATGCGGCGCTCGGTCATCGCTGCATTGATCGAGTAGCCACCGCCGCCGCAAATGCCCAGCACGCCAATGCGTTCTTCATCCACATAGGGGAGCGTTACGAGGTAGTCGCAGACCACGCGGAAATCCTCGACGCGCAGTGTCGGGTCTTCGGTAAAACGTGGTTCGCCGCCGCTGGCACCCTGGAAGCTGGCGTCGAAGGCGATGACGACGAAACCTTCCTTGGCCAGCGCCTCGCCATACACGTTCCCCGATGTTTGCTCCTTGCAGCTGCCGATCGGATGCGCGCTGATGATGGCGGGATATTTCTTGCCTTCGTCGAAGTTCGGCGGGAAGTGGATGTCGGCTGCGATATCCCAATACACATTCTTGATCTTGACGCTTTTCATGACAGCTCCGTTCAGGGGGAGGGGGTAAGTTCGCCAGGCCGAATCGTTGGTAGCCAAGCGGCAACGACTCGAATATAGAGAGCCGATTGGAATTCCGTAAGATCGCAATCTGGACTACAGTGGTATCTTCAAATTGACAATGGCACCTACATGGATCCCTCACTGCTTCCGTCTCTCGCGTGGTTCGCCCACGTCGCACATCATCGTAGCTTCACGAAAGCGGCTGCGGAAATGGGCGTTTCTCGAGCAAACCTGTCGCAGAACGTGAAGGCGCTCGAACGCCGGTTGAACGTCAAGCTGCTGTATCGAACGACTCGCGACATG

25

In [115]:
def number_of_seqs(file_name):
    '''This function returns a count of the number of sequences in a FASTA file.

    Args:

        file_name (str): Path of the FASTA file.

    Returns:    
        int: Number of sequences in the file.
    '''
    dna_seqs = extract_dna_seqs(file_name)
    count = 0
    for line in dna_seqs:
        if line.startswith(">"):
            count += 1
            
    return count

count = number_of_seqs("dna.example.fasta")
print(count)



25


(2) What are the lengths of the sequences in the file? What is the longest sequence and what is the shortest sequence? Is there more than one longest or shortest sequence? What are their identifiers? 

In [116]:
def length_seqs_stats(file_name, output="general", print_output = False):
    '''This function provides some simple statistics of the lengths of the sequences in a FASTA file.
    
    Args:

        file_name (str): Path of the FASTA file.

        output (str): Alters the return value of the function and its possible print
        into console. "general" by default. One of:
            - "general": This returns a dictionary of sequence lengths along with their 
                        identifiers, a dictionary of the longest sequence(s) along with
                        their identifiers, the length of the longest sequence, a
                        similar dictionary for the shortest sequence(s), and the length
                        of the shortest sequence.
            
            - "lengths": This returns a dictionary of sequence lengths along with their 
                        identifiers.

        print_output (bool): Boolean value that decides if function prints into console. False by
        default.
    
    Returns:
        tuple or dict:
            - If output == "general": a 5-item tuple of
            (lengths_id_dict, longest_seqs, longest_seq_len,
            shortest_seqs, shortest_seq_len).
            - If output == "lengths": a dict mapping sequence id to
            sequence length.
    
    Raises:
        ValueError: If unspecified value for output is inputted.
    '''
    dna_seqs_dict = create_dna_dict(file_name)

    lengths_id_dict = {seq_id:len(seq) for seq_id,seq in dna_seqs_dict.items()}

    longest_seq_len = max(lengths_id_dict.values())
    shortest_seq_len = min(lengths_id_dict.values())
    longest_seqs = {seq_id:seq for seq_id,seq in dna_seqs_dict.items() if len(seq) == longest_seq_len}
    shortest_seqs = {seq_id:seq for seq_id,seq in dna_seqs_dict.items() if len(seq) == shortest_seq_len}

    if output == "general":
        if print_output:
            print("General statistics of the provided DNA sequences:\n")
            print(f"Longest sequence(s):      Length: {longest_seq_len}")
            for seq_id, seq in longest_seqs.items():
                print(seq_id)
                print(seq)
                print()

            print()
            print(f"Shortest sequence(s):     Length: {shortest_seq_len}")
            for seq_id, seq in shortest_seqs.items():
                print(seq_id)
                print(seq)
                print()

        return_value = lengths_id_dict, longest_seqs, longest_seq_len, shortest_seqs, shortest_seq_len
        
    elif output == "lengths":     
        if print_output:
            print("Lengths of sequences:")
            for seq_id, length in lengths_id_dict.items():
                print(seq_id)
                print(length)

        return_value = lengths_id_dict

    else:
        raise ValueError(f"Unknown output type: {output}")

    return return_value
        
result = length_seqs_stats(file_name="dna.example.fasta", print_output=True)       

General statistics of the provided DNA sequences:

Longest sequence(s):      Length: 4805
gi|142022655|gb|EQ086233.1|323 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence
ACGCCCGGCGCACCGCGAGTACCGCGCCGCCGGGCACTCCTTGACCCCGCATGATCGATTCCCGATGAAACCCGAAAACCTCGTCGCCTGCCACGAATGCGACCTGCTGTTTTGGCGGCCGCCGCGCTTGCGCGCGCTGGCTGCGCACTGCCCGAGGTGCCGTGCCCGCGTGGGCGGCAGCGCGCACGGCCGTCCGGCGCTCGACCGGCGGTGCGCGATCGCGCTCGCCGCGCTGTTCACGCTCTTCATCGCGCAGGCCTTTCCCATCGTCGCGCTCGACGCCGCCGGCATCGCATCGCACGCGACGCTGGCCGACGCGGTGGCCGCGTTGCGCTTGAACGGGCAACCGGCGGTGGCGGCGATCGTGTTCTGCACGACGATGTTGTTCCCGCTGCTGGAACTCGCCGCGTGGCTGTACGTGCTCGTACCGTTGCGCGCGGGCCGCGTACCGCCCCGCTTCGAGCCGGTCCTGCGCAACATGCAGCGGCTGCGCCCGTGGAGCATGGTCGAGGTGTTCCTGCTCGGCATCCTGGTCACGATCGTCAAGATGACGAGCCTCGCGCACGTGATACCGGGCCCCGCGCTGTTTGCGTTCGGCGCCCTCACCGTGTTGCTCGGCTTTCTCGCGTCATTCGACCCGGGCGGCCTGTGGGAAGCGCGCGACGAAATCATCGCGCTGCGCGGCGGCGGTACGTCCGCCGCGGTATCGCGCCGGCGGCACACGCCGCGACGCGCTGCACCGGTGACGCCCGACACAGCGGACGCAACGAACGCGACCGGCGCGACCGG

(3) In molecular biology, a reading frame is a way of dividing the DNA sequence of nucleotides into a set of consecutive, non-overlapping triplets (or codons). Depending on where we start, there are six possible reading frames: three in the forward (5' to 3') direction and three in the reverse (3' to 5'). For instance, the three possible forward reading frames for the sequence AGGTGACACCGCAAGCCTTATATTAGC are: 

AGG TGA CAC CGC AAG CCT TAT ATT AGC

A GGT GAC ACC GCA AGC CTT ATA TTA GC

AG GTG ACA CCG CAA GCC TTA TAT TAG C 

These are called reading frames 1, 2, and 3 respectively. An open reading frame (ORF) is the part of a reading frame that has the potential to encode a protein. It starts with a start codon (ATG), and ends with a stop codon (TAA, TAG or TGA). For instance, ATGAAATAG is an ORF of length 9.

Given an input reading frame on the forward strand (1, 2, or 3) your program should be able to identify all ORFs present in each sequence of the FASTA file, and answer the following questions: what is the length of the longest ORF in the file? What is the identifier of the sequence containing the longest ORF? For a given sequence identifier, what is the longest ORF contained in the sequence represented by that identifier? What is the starting position of the longest ORF in the sequence that contains it? The position should indicate the character number in the sequence. For instance, the following ORF in reading frame 1:

\>sequence1

ATGCCCTAG

starts at position 1.

Note that because the following sequence:

\>sequence2

ATGAAAAAA

does not have any stop codon in reading frame 1, we do not consider it to be an ORF in reading frame 1.

In [117]:
def seq_rf(seq, reading_frame_pos=1):
    if reading_frame_pos not in (1, 2, 3):
        raise ValueError("Incompatible value inputted for reading_frame parameter")

    i = reading_frame_pos - 1
    if reading_frame_pos > 1:
        codons = [seq[:i]]
    else:
        codons = []

    while i + 3 <= len(seq):
        codons.append(seq[i:i + 3])
        i += 3

    if i != len(seq):
        codons.append(seq[i:])

    return codons

def orf(file_name, reading_frame_pos=1):
    dna_seq_dict = create_dna_dict(file_name)
    seq_orfs_dict = {}
    rf_pos_list = [1, -1, 0]
    rf_pos = rf_pos_list[rf - 1]

    for seq_id, seq in dna_seq_dict.items():
        reading_frame = seq_rf(seq, reading_frame_pos)

        start_codon_indices = [
            index
            for index, value in enumerate(reading_frame)
            if value == "ATG"
        ]
        stop_codon_indices = [
            index
            for index, value in enumerate(reading_frame)
            if value in ("TAA", "TAG", "TGA")
        ]

        prev_stop_codon = 0
        orfs_for_curr_seq = {}

        for stop_codon_index in stop_codon_indices:
            # This finds all start codons before the current
            # stop codon after the previous stop codon allowing
            # me to isolate the possible ORF(s).
            start_codon_indices_before = [
                start_codon_index
                for start_codon_index in start_codon_indices
                if prev_stop_codon <= start_codon_index < stop_codon_index
            ]

            for scib in start_codon_indices_before:
                orfs_for_curr_seq[f"{scib * 3 + rf_pos}-{stop_codon_index * 3 + rf_pos + 2}"] = "".join(
                    reading_frame[scib:stop_codon_index + 1]
                )

            prev_stop_codon = stop_codon_index

        seq_orfs_dict[seq_id] = orfs_for_curr_seq

    return seq_orfs_dict

seq_orfs_dict = orf("dna.example.fasta")
pprint(seq_orfs_dict)

{'gi|142022655|gb|EQ086233.1|101 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1021-1131': 'ATGCCCGAGACGATCGTGCAGAACACGATCGGCGCGATGGTCATCCTCACGAGGCCGACGAACGCGTCGCTGAGCGGTTTGAACATCGCGCCTGCGTCCGGCCATACATGA',
                                                                                                                              '1057-1131': 'ATGGTCATCCTCACGAGGCCGACGAACGCGTCGCTGAGCGGTTTGAACATCGCGCCTGCGTCCGGCCATACATGA',
                                                                                                                              '121-282': 'ATGACGACGAGCGTGGCGACCAGCGCAACCAGCCCGCTTCCGGAAACGCCGGCCGCGCCCTTGGACGTGAGCAGCATGATGGCGAGCATCACGGCGATCTGCGACGCGGAAAGGGGCACGTCGCACGCCTGCGCGATGAACAACGCGGCGAGCGTCAGATAG',
                                                                                                                              '1633-2271': 'ATGGTCGCGGCGGCAGGCAGCGACGAACCGCTCCTGGAGCAACATCTTGAACTCGATGTCGGATTCCTGGCTGCCCATGAAGCTC

In [118]:
longest_per_seq_orfs = {
    seq_id:{
        orf_pos:max_len_orf
        for orf_pos, max_len_orf in orfs_dict.items()
        if len(max_len_orf) == len(max(orfs_dict.values(), key=len))
    }
    for seq_id, orfs_dict in seq_orfs_dict.items()
}

pprint(longest_per_seq_orfs)

{'gi|142022655|gb|EQ086233.1|101 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'1633-2271': 'ATGGTCGCGGCGGCAGGCAGCGACGAACCGCTCCTGGAGCAACATCTTGAACTCGATGTCGGATTCCTGGCTGCCCATGAAGCTCACGCCGAAATCGGCTTCGCCGCTGATGACGGCGCCCAGCACCTCGTTCGCGCTCGCGTCCAGCAGCTTGACCCGGATGCGCGGAAAGCGCTGATGATAGCGCGCGATGATGGCCGGCAGAAAGTAGTAGGCGACCGAGGGCACGCACGCGATGGTCACATGGCCCAGGCGGCTCGACGACACGTCGCGAATGCCGAGCAGCGCCGCATCGAGATCGTCGAGCAGCTGTTCGGCGCTCTGGGCGAACACGCGGCCGACCGTGGTGAGCGCGACGCGACGCGTGGTGCGCTCGAACAGGCGCACGCCGAGCGCTTCCTCGAGCTTGTCGATCCGGCGACTCAACGCGGGCTGGGAAATGCTGACCGATTCCGCGGCCTTGCGGAAACTGCCCGTTTCCACGACCGCGCGAAACGCCTGCAAGTCGTTCAAGTCGAAGTTGATCCCCACGGGCGCGTCTCCCCATCTCAGATGGGGCGTATTTTGCATGATTTCGCCGGGCGGCCGCATCGGCGCGGCACGCATTCGCGCCACCCTCGATCGCAACCGCGTGCGTGA'},
 'gi|142022655|gb|EQ086233.1|158 marine metagenome JCVI_SCAF_1096627390048 genomic scaffold, whole genome shotgun sequence': {'700-828': 'ATGCCGTCCAGCTTGCCGTGGTCGTGGCGGCGCTGTTTTTCGGCGCGTTCATGCATCCTGCCGTCAACACAAGCGAGGA

In [ ]:
longest_orf_length = 0
for longestper_orf_dict in longest_per_seq_orfs.values():
    if longestper_orf_dict == {}:
        continue
    elif len(list(longestper_orf_dict.values())[0]) >= longest_orf_length:
        longest_orf_length = len(longestper_orf_dict.values()[0])

longest_orf = {
    seq_id:longest_orf_dict
    for seq_id, longest_orf_dict in longest_per_seq_orfs.items()
    if len(longest_orf_dict.values()[0]) == longest_orf_length
}
        
print(longest_orf)

TypeError: 'dict_values' object is not subscriptable

: 

In [ ]:
r = {'r':'ready','d':'dog', 'z':'hippopotamus'}
print(r)
length_dict = {seq_id:len(seq) for seq_id,seq in r.items()}
print(length_dict)
print(max(length_dict.values()))
print(list(length_dict.values()))

{'r': 'ready', 'd': 'dog', 'z': 'hippopotamus'}
{'r': 5, 'd': 3, 'z': 12}
12
[5, 3, 12]


In [ ]:
sequence1 = "ATGATGTAAATGATGTGAGATGTGACATGATGTAG"
print(sequence1)

rf = 1

i = rf - 1
if rf > 1:
    codons = [sequence1[:i]]
else:
    codons = []

while i+3 <= len(sequence1):
    codons.append(sequence1[i:i+3])
    i += 3

if i != len(sequence1):
    codons.append(sequence1[i:])
print(codons)

rfs = seq_rf(sequence1, rf)
print(rfs)

start_codon_indices1 = [
    index
    for index, value in enumerate(rfs)
    if value == 'ATG'
    ]
print(f"Start codon positions: {start_codon_indices1}")

stop_codon_indices1 = [
    index
    for index, value in enumerate(rfs)
    if value in ('TAG', 'TAA', 'TGA')
]
print(f"Stop codons positions: {stop_codon_indices1}")

prev_stop_codon = 0
open_rfs = {}
pos_list = [1, -1, 0]

for stop_codon_index in stop_codon_indices1:
    start_codon_indices_before1 = [
        start_codon_index
        for start_codon_index in start_codon_indices1
        if (prev_stop_codon <= start_codon_index < stop_codon_index)
    ]

    for scib in start_codon_indices_before1:
        open_rfs[f"{scib * 3 + pos_list[rf - 1]}-{stop_codon_index * 3 + pos_list[rf - 1] + 2}"] = "".join(rfs[scib:stop_codon_index + 1])
        # print(f"{scib}-{stop_codon_index}: {"".join(rfs[scib:stop_codon_index + 1])}")

    # print(open_rfs)    
    prev_stop_codon = stop_codon_index
print()
pprint(open_rfs)
seq_orfs =  {}
seq_orfs[sequence1] = open_rfs
pprint(seq_orfs)

ATGATGTAAATGATGTGAGATGTGACATGATGTAG
['ATG', 'ATG', 'TAA', 'ATG', 'ATG', 'TGA', 'GAT', 'GTG', 'ACA', 'TGA', 'TGT', 'AG']
['ATG', 'ATG', 'TAA', 'ATG', 'ATG', 'TGA', 'GAT', 'GTG', 'ACA', 'TGA', 'TGT', 'AG']
Start codon positions: [0, 1, 3, 4]
Stop codons positions: [2, 5, 9]

{'1-9': 'ATGATGTAA', '10-18': 'ATGATGTGA', '13-18': 'ATGTGA', '4-9': 'ATGTAA'}
{'ATGATGTAAATGATGTGAGATGTGACATGATGTAG': {'1-9': 'ATGATGTAA',
                                         '10-18': 'ATGATGTGA',
                                         '13-18': 'ATGTGA',
                                         '4-9': 'ATGTAA'}}


In [ ]:
seq_len_orfs = {
    seq_id:{
        pos:max_len_orf
        for pos, max_len_orf in orfs_dict.items()
        if len(max_len_orf) == len(max(orfs_dict.values(), key=len))}
    for seq_id, orfs_dict in seq_orfs.items()
}
print(seq_len_orfs)

{'ATGATGTAAATGATGTGAGATGTGACATGATGTAG': {'1-9': 'ATGATGTAA', '10-18': 'ATGATGTGA'}}
